## Import Libraries and Datasets

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

DATA_DIR = Path("data")
sorted(p.name for p in DATA_DIR.iterdir())

['bars_1m.parquet',
 'events.json',
 'exchange_calendar.csv',
 'regime_labels.csv',
 'sim_runs.json',
 'stream_bars.json',
 'symbol_history.csv',
 'symbols.csv',
 'vendor_daily.csv']

In [2]:
exchange_calendar = pd.read_csv(DATA_DIR / "exchange_calendar.csv")
symbol_history = pd.read_csv(DATA_DIR / "symbol_history.csv")
symbols = pd.read_csv(DATA_DIR / "symbols.csv")
vendor_daily = pd.read_csv(DATA_DIR / "vendor_daily.csv")
bars = pd.read_parquet(DATA_DIR / "bars_1m.parquet")

exchange_calendar["_source_row"] = range(len(exchange_calendar))
symbol_history["_source_row"] = range(len(symbol_history))
symbols["_source_row"] = range(len(symbols))
vendor_daily["_source_row"] = range(len(vendor_daily))
bars["_source_row"] = range(len(bars))

for name, df in [
    ("exchange_calendar", exchange_calendar),
    ("symbol_history", symbol_history),
    ("symbols", symbols),
    ("vendor_daily", vendor_daily),
    ("bars", bars),
]:
    print(f"{name:16s} {df.shape}  {list(df.columns)[:6]} ...")

exchange_calendar (129, 7)  ['date', 'is_trading_day', 'session_open', 'session_close', 'timezone', 'note'] ...
symbol_history   (3, 6)  ['security_id', 'change_type', 'old_value', 'new_value', 'effective_date', '_source_row'] ...
symbols          (31, 8)  ['security_id', 'ticker', 'company_name', 'sector', 'listing_date', 'delisting_date'] ...
vendor_daily     (10831, 9)  ['run_id', 'date', 'ticker', 'open', 'high', 'low'] ...
bars             (4276175, 10)  ['run_id', 'ticker', 'ts', 'open', 'high', 'low'] ...


## Flatten nested JSON

In [3]:
import json

with open(DATA_DIR / "sim_runs.json", encoding="utf-8") as f:
    sim_runs_raw = json.load(f)

with open(DATA_DIR / "events.json", encoding="utf-8") as f:
    events_raw = json.load(f)

In [4]:
sim_runs_flat = pd.json_normalize(sim_runs_raw, sep="_")
sim_runs_flat.insert(0, "source_row", range(1, len(sim_runs_flat) + 1))

events_flat = pd.json_normalize(events_raw["events"], sep="_")
events_flat.insert(0, "source_row", range(1, len(events_flat) + 1))

In [5]:
print("SIM_RUNS")
print(sim_runs_flat.shape)
print(sim_runs_flat.columns.tolist())
display(sim_runs_flat.head())

print("\nEVENTS")
print(events_flat.shape)
print(events_flat.columns.tolist())
display(events_flat.head())

SIM_RUNS
(4, 25)
['source_row', 'run_id', 'config_version', 'created_at', 'seed', 'window_start', 'window_end', 'universe_stocks', 'universe_index_ticker', 'universe_index_method', 'universe_index_base_level', 'parameters_volatility_sigma_annual', 'parameters_climate_states', 'parameters_climate_variance_multiplier', 'parameters_climate_transition_matrix', 'parameters_news_memory_alpha', 'parameters_news_memory_beta', 'parameters_news_memory_gamma', 'parameters_intraday_open_boost', 'parameters_intraday_close_boost', 'parameters_shocks_distribution', 'parameters_shocks_df', 'parameters_jumps_rate_per_day', 'parameters_factors_market_weight', 'parameters_factors_sector_share']


,source_row,run_id,config_version,created_at,seed,window_start,window_end,universe_stocks,universe_index_ticker,universe_index_method,...,parameters_news_memory_alpha,parameters_news_memory_beta,parameters_news_memory_gamma,parameters_intraday_open_boost,parameters_intraday_close_boost,parameters_shocks_distribution,parameters_shocks_df,parameters_jumps_rate_per_day,parameters_factors_market_weight,parameters_factors_sector_share
0,1,RUN-2025A,1.4.0,2025-07-02T08:00:00+02:00,8522640,2025-01-02,2025-06-30,30,INDEX,cap_weighted,...,0.03,0.8,0.6,1.2,0.6,student_t,4,0.02,"[0.12, 0.22, 0.45]",0.25
1,2,RUN-2025B,1.4.1,2025-07-02T08:00:00+02:00,7737823,2025-01-02,2025-06-30,30,INDEX,cap_weighted,...,0.33,0.8,0.0,0.0,0.0,student_t,4,0.02,"[0.12, 0.22, 0.45]",0.25
2,3,RUN-2025C,1.4.0,2025-07-02T08:00:00+02:00,8325121,2025-01-02,2025-06-30,30,INDEX,cap_weighted,...,0.03,0.8,0.6,1.2,0.6,student_t,4,0.02,"[0.12, 0.22, 0.45]",0.25
3,4,RUN-2025B,1.4.1,2025-07-03T11:20:00+02:00,7737823,2025-01-02,2025-06-30,30,INDEX,cap_weighted,...,0.33,0.8,0.0,0.0,0.0,student_t,4,0.02,"[0.12, 0.22, 0.45]",0.25



EVENTS
(62, 8)
['source_row', 'event_id', 'type', 'time', 'security_ticker', 'details_session', 'details_duration_minutes', 'details_ratio']


,source_row,event_id,type,time,security_ticker,details_session,details_duration_minutes,details_ratio
0,1,EV0001,earnings,2025-01-10T09:30:00-05:00,DALN,pre_open,NaN,NaN
1,2,EV0002,earnings,2025-01-14T09:30:00-05:00,BRIS,pre_open,NaN,NaN
2,3,EV0003,earnings,2025-01-15T09:30:00-05:00,SOLK,pre_open,NaN,NaN
3,4,EV0004,earnings,2025-01-24T09:30:00-05:00,XERN,pre_open,NaN,NaN
4,5,EV0005,earnings,2025-01-30T09:30:00-05:00,VALX,pre_open,NaN,NaN


In [6]:
bars.head()

,run_id,ticker,ts,open,high,low,close,volume,received_at,_source_row
0,RUN-2025A,ARVT,2025-01-02 09:30:00,120.07,120.08,119.84,119.93,8570,2025-01-02 14:31:00+00:00,0
1,RUN-2025A,ARVT,2025-01-02 09:31:00,119.93,119.98,119.35,119.39,45001,2025-01-02 14:32:01+00:00,1
2,RUN-2025A,ARVT,2025-01-02 09:32:00,119.39,119.66,119.30,119.53,5825,2025-01-02 14:33:00+00:00,2
3,RUN-2025A,ARVT,2025-01-02 09:33:00,119.53,119.57,119.23,119.35,8195,2025-01-02 14:34:00+00:00,3
4,RUN-2025A,ARVT,2025-01-02 09:34:00,119.35,119.44,119.33,119.38,5631,2025-01-02 14:35:01+00:00,4


In [7]:
sim_runs_flat.head()

,source_row,run_id,config_version,created_at,seed,window_start,window_end,universe_stocks,universe_index_ticker,universe_index_method,...,parameters_news_memory_alpha,parameters_news_memory_beta,parameters_news_memory_gamma,parameters_intraday_open_boost,parameters_intraday_close_boost,parameters_shocks_distribution,parameters_shocks_df,parameters_jumps_rate_per_day,parameters_factors_market_weight,parameters_factors_sector_share
0,1,RUN-2025A,1.4.0,2025-07-02T08:00:00+02:00,8522640,2025-01-02,2025-06-30,30,INDEX,cap_weighted,...,0.03,0.8,0.6,1.2,0.6,student_t,4,0.02,"[0.12, 0.22, 0.45]",0.25
1,2,RUN-2025B,1.4.1,2025-07-02T08:00:00+02:00,7737823,2025-01-02,2025-06-30,30,INDEX,cap_weighted,...,0.33,0.8,0.0,0.0,0.0,student_t,4,0.02,"[0.12, 0.22, 0.45]",0.25
2,3,RUN-2025C,1.4.0,2025-07-02T08:00:00+02:00,8325121,2025-01-02,2025-06-30,30,INDEX,cap_weighted,...,0.03,0.8,0.6,1.2,0.6,student_t,4,0.02,"[0.12, 0.22, 0.45]",0.25
3,4,RUN-2025B,1.4.1,2025-07-03T11:20:00+02:00,7737823,2025-01-02,2025-06-30,30,INDEX,cap_weighted,...,0.33,0.8,0.0,0.0,0.0,student_t,4,0.02,"[0.12, 0.22, 0.45]",0.25


### Rejection Function

In [8]:
# Rejection 
def make_rejects(df, mask, reason, source_file):
    bad_rows = df.loc[mask]

    reject_rows = [
        {
            "raw_payload": row.drop(labels="_source_row").to_dict(),
            "source_file": source_file,
            "source_row": int(row["_source_row"]),
            "reason": reason,
        }
        for _, row in bad_rows.iterrows()
    ]

    return pd.DataFrame(reject_rows)


all_rejects = []


In [9]:
exchange_calendar.head()

,date,is_trading_day,session_open,session_close,timezone,note,_source_row
0,2025-01-02,True,09:30,16:00,America/New_York,NaN,0
1,2025-01-03,True,09:30,16:00,America/New_York,NaN,1
2,2025-01-06,True,09:30,16:00,America/New_York,NaN,2
3,2025-01-07,True,09:30,16:00,America/New_York,NaN,3
4,2025-01-08,True,09:30,16:00,America/New_York,NaN,4


### EDA `bar_1m`

In [10]:
bars_cleaned = bars.copy()

In [11]:
bars.describe().style.format("{:,.2f}")

,open,high,low,close,volume,_source_row
count,"4,276,175.00","4,276,175.00","4,276,175.00","4,275,321.00","4,276,175.00","4,276,175.00"
mean,149.36,149.51,149.21,149.36,"8,358.85","2,138,087.00"
std,183.48,183.60,183.36,183.48,"25,639.99","1,234,425.54"
min,4.81,4.84,4.80,4.81,"-47,568.00",0.00
25%,63.54,63.59,63.48,63.54,"1,599.00","1,069,043.50"
50%,113.11,113.25,112.97,113.11,"2,825.00","2,138,087.00"
75%,157.54,157.75,157.35,157.54,"5,232.00","3,207,130.50"
max,"1,483.31","1,483.84","1,482.76","1,483.31","13,208,790.00","4,276,174.00"


#### Missing Values

In [12]:
bars.isna().sum()

run_id           0
ticker           0
ts               0
open             0
high             0
low              0
close          854
volume           0
received_at      0
_source_row      0
dtype: int64

In [13]:
mask = bars["close"].isna()
all_rejects.append(make_rejects(bars, mask, "MISSING_REQUIRED_FIELD", "bars_1m.parquet"))
bars_cleaned = bars.loc[~mask]

##### Negative Values

In [14]:
#-ve values
negative_value=(bars[["open","high","low","close","volume"]] <= 0).any(axis=1) #  at least one value in this row True
negative_value.sum()

np.int64(853)

In [15]:
mask = negative_value
all_rejects.append(make_rejects(bars, mask, "INVALID_NUMERIC_RANGE", "bars_1m.parquet"))
bars_cleaned = bars_cleaned.loc[~mask]

#### Invalid OHLC

In [16]:
# OHLC consistency (high must be highest, low must be lowest)
bad_ohlc = (
    (bars.high < bars.low) |
    (bars.high < bars[["open","close"]].max(axis=1)) |
    (bars.low  > bars[["open","close"]].min(axis=1))
)
bad_ohlc.sum()

np.int64(1282)

In [17]:
# Invalid OHLC
mask = bad_ohlc
all_rejects.append(make_rejects(bars, mask, "INVALID_OHLC", "bars_1m.parquet"))
bars_cleaned = bars_cleaned.loc[~mask]


#### Duplicates

In [18]:
# duplicated records
bars.duplicated(subset=["run_id","ticker","ts"]).sum()

np.int64(15120)

In [19]:
# Duplicates (keep the earliest received_at)
bars = bars.sort_values("received_at")
mask = bars.duplicated(["run_id", "ticker", "ts"], keep="first")
all_rejects.append(make_rejects(bars, mask, "DUPLICATE_BAR", "bars_1m.parquet"))
bars_cleaned = bars_cleaned.loc[~mask]

In [20]:
bars["run_id"].value_counts()

run_id
RUN-2025A    1431358
RUN-2025B    1424991
RUN-2025C    1419566
RUN-TEST         260
Name: count, dtype: int64

### Bars with an unknown run

**Finding:** The bars table contains 260 rows with `run_id = RUN-TEST`. This ID does not appear in the flattened `sim_runs` table.

**How it was detected:** I compared the distinct `run_id` values in `bars` with those in `sim_runs_flat`. The `RUN-TEST` bars did not match a known simulation run.

**Decision:** These bars are candidates for rejection because their run cannot be verified. For each rejected bar, the rejects table preserves the raw bar payload, source file (`bars_1m.parquet`), source row, and one rejection reason: `UNKNOWN_RUN`.

In [21]:
mask = bars["run_id"] == "RUN-TEST"
all_rejects.append(make_rejects(bars, mask, "UNKNOWN_RUN", "bars_1m.parquet"))
bars_cleaned = bars_cleaned.loc[~mask]

#### Fake Nulls

In [22]:
FAKE_NULLS = {"", "null", "none", "nan", "na", "n/a", "-", "?"}

for col in ["run_id", "ticker", "ts"]:
    s = bars[col].astype("string").str.strip().str.lower()
    print(col, s.isin(FAKE_NULLS).sum())

run_id 0
ticker 0
ts 0


#### Timestamp Check

In [23]:
bars["ts"].str.len().value_counts()

ts
19    2851052
20    1424849
16        159
10        115
Name: count, dtype: int64

In [24]:
bars.assign(ts_len=bars["ts"].str.len()).groupby("ts_len", dropna=False).head(3).sort_values("ts_len")

,run_id,ticker,ts,open,high,low,close,volume,received_at,_source_row,ts_len
578109,RUN-2025A,MARB,not-a-time,145.35,145.42,145.23,145.26,3640,2025-01-02 15:46:01+00:00,578109,10
1377917,RUN-2025A,INDEX,not-a-time,988.70,988.75,988.34,988.54,129105,2025-01-02 16:31:02+00:00,1377917,10
1911069,RUN-2025B,KELT,not-a-time,88.05,88.10,87.98,88.01,3728,2025-01-02 17:07:01+00:00,1911069,10
1767283,RUN-2025B,HALS,31/03/2025 10:15,58.70,58.73,58.69,58.71,4639,2025-01-02 18:47:01+00:00,1767283,16
2662373,RUN-2025B,ASTR,31/03/2025 10:15,162.12,162.14,162.02,162.02,5536,2025-01-03 16:12:02+00:00,2662373,16
2534653,RUN-2025B,XERN,31/03/2025 10:15,17.78,17.78,17.77,17.77,3824,2025-01-02 19:03:02+00:00,2534653,16
0,RUN-2025A,ARVT,2025-01-02 09:30:00,120.07,120.08,119.84,119.93,8570,2025-01-02 14:31:00+00:00,0,19
578034,RUN-2025A,MARB,2025-01-02 09:30:00,146.30,146.42,146.29,146.38,4849,2025-01-02 14:31:00+00:00,578034,19
3047740,RUN-2025C,ERTL,2025-01-02 09:30:00,117.22,117.81,117.10,117.46,7116,2025-01-02 14:31:00+00:00,3047740,19
2438467,RUN-2025B,VALX,2025-01-02T14:30:00Z,227.66,227.90,227.60,227.79,4109,2025-01-02 14:31:00+00:00,2438467,20


The timestamps `ts` use multiple date formats, standardize them to match the `exchange_calendar`

In [25]:
exchange_calendar.timezone.unique()

<ArrowStringArray>
['America/New_York']
Length: 1, dtype: str

In [26]:
exchange_calendar.head()

,date,is_trading_day,session_open,session_close,timezone,note,_source_row
0,2025-01-02,True,09:30,16:00,America/New_York,NaN,0
1,2025-01-03,True,09:30,16:00,America/New_York,NaN,1
2,2025-01-06,True,09:30,16:00,America/New_York,NaN,2
3,2025-01-07,True,09:30,16:00,America/New_York,NaN,3
4,2025-01-08,True,09:30,16:00,America/New_York,NaN,4


In [27]:
TZ = "America/New_York"    # based on exchange_calendar.csv (timezone column)

ts = bars["ts"].astype("string").str.strip()

# Format A: 2025-01-02 09:30:00  (already local)
fmt_local = pd.to_datetime(ts, format="%Y-%m-%d %H:%M:%S", errors="coerce")

# Format B: 2025-01-02T14:30:00Z  (UTC → convert to local)
fmt_utc = pd.to_datetime(ts, format="%Y-%m-%dT%H:%M:%SZ", errors="coerce", utc=True)
fmt_utc = fmt_utc.dt.tz_convert(TZ).dt.tz_localize(None)

# Format C: 31/03/2025 10:15  (day/month/year) 
fmt_dmy = pd.to_datetime(ts, format="%d/%m/%Y %H:%M", errors="coerce")

# Combine: take the first format that worked
bars_cleaned["ts_local"] = fmt_local.fillna(fmt_utc).fillna(fmt_dmy)

# Remember what was converted (quality flag)
bars["ts_local"] = fmt_local.fillna(fmt_utc).fillna(fmt_dmy)
bars["ts_flag"] = None
bars.loc[fmt_utc.notna(), "ts_flag"] = "TS_CONVERTED_FROM_UTC"
bars.loc[fmt_dmy.notna(), "ts_flag"] = "TS_REFORMATTED_DMY"

In [28]:
# Fixed dates
display(bars_cleaned.loc[bars_cleaned["ts_local"].isna(), "ts"].value_counts())

ts
2025-02-30 10:00:00    79
not-a-time             63
Name: count, dtype: int64

In [29]:
mask = bars["ts_local"].isna()
all_rejects.append(make_rejects(bars, mask, "MALFORMED_TIMESTAMP", "bars_1m.parquet"))
bars_cleaned = bars_cleaned.loc[~mask]

In [30]:
# 2. compare with received_at: lateness should be small and positive for every format

bars_cleaned["end_utc"] = (bars_cleaned["ts_local"] + pd.Timedelta(minutes=1)) \
            .dt.tz_localize(TZ, ambiguous="NaT", nonexistent="NaT") \
            .dt.tz_convert("UTC")                        # 2. safe around clock changes

bars_cleaned["lateness_s"] = (bars_cleaned["received_at"] - bars_cleaned["end_utc"]).dt.total_seconds()

ts_len = bars_cleaned["ts"].astype("string").str.strip().str.len()   # 3. computed here, so it always exists
display(bars_cleaned["lateness_s"].groupby(ts_len).median())

ts
16   -3099238.0
19          1.0
20          1.0
Name: lateness_s, dtype: float64

In [31]:
# Placeholder 
g = bars_cleaned[ts_len == 16]
g["ts"].value_counts().head(10)


ts
31/03/2025 10:15    76
Name: count, dtype: int64

In [32]:
g

,run_id,ticker,ts,open,high,low,close,volume,received_at,_source_row,ts_local,end_utc,lateness_s
9247,RUN-2025A,ARVT,31/03/2025 10:15,134.22,134.33,133.73,133.83,10727,2025-02-04 19:05:01+00:00,9247,2025-03-31 10:15:00,2025-03-31 14:16:00+00:00,-4734659.0
152057,RUN-2025A,DALN,31/03/2025 10:15,181.95,183.10,181.94,182.50,22583,2025-01-29 16:29:01+00:00,152057,2025-03-31 10:15:00,2025-03-31 14:16:00+00:00,-5262419.0
231732,RUN-2025A,ERTL,31/03/2025 10:15,78.05,78.06,77.98,78.03,3177,2025-05-27 17:07:01+00:00,231732,2025-03-31 10:15:00,2025-03-31 14:16:00+00:00,4935061.0
275995,RUN-2025A,FENM,31/03/2025 10:15,63.15,63.16,63.12,63.12,1781,2025-05-12 17:00:00+00:00,275995,2025-03-31 10:15:00,2025-03-31 14:16:00+00:00,3638640.0
291246,RUN-2025A,GALR,31/03/2025 10:15,20.61,20.63,20.57,20.58,2780,2025-01-09 18:45:01+00:00,291246,2025-03-31 10:15:00,2025-03-31 14:16:00+00:00,-6982259.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4041977,RUN-2025C,ZENM,31/03/2025 10:15,50.93,51.10,50.65,50.87,941,2025-01-29 19:23:02+00:00,4041977,2025-03-31 10:15:00,2025-03-31 14:16:00+00:00,-5251978.0
4108862,RUN-2025C,ASTR,31/03/2025 10:15,218.35,218.35,218.27,218.34,2161,2025-04-09 18:21:00+00:00,4108862,2025-03-31 10:15:00,2025-03-31 14:16:00+00:00,792300.0
4137627,RUN-2025C,BRIS,31/03/2025 10:15,81.96,83.02,81.89,82.66,2294,2025-01-29 20:54:02+00:00,4137627,2025-03-31 10:15:00,2025-03-31 14:16:00+00:00,-5246518.0
4196757,RUN-2025C,CELK,31/03/2025 10:15,132.00,132.02,131.73,131.81,1654,2025-05-09 17:06:01+00:00,4196757,2025-03-31 10:15:00,2025-03-31 14:16:00+00:00,3379801.0


In [33]:
placeholder = bars_cleaned["ts"].astype("string").str.strip() == "31/03/2025 10:15"
mask = placeholder
all_rejects.append(make_rejects(bars_cleaned, mask, "MALFORMED_TIMESTAMP", "bars_1m.parquet"))
bars_cleaned = bars_cleaned.loc[~mask]
print(mask.sum())    # 159

76


In [34]:
## Adding columns
bars_cleaned["trading_date"] = bars_cleaned["ts_local"].dt.date
bars_cleaned["bar_time"]     = bars_cleaned["ts_local"].dt.time

In [35]:
bars_cleaned

,run_id,ticker,ts,open,high,low,close,volume,received_at,_source_row,ts_local,end_utc,lateness_s,trading_date,bar_time
0,RUN-2025A,ARVT,2025-01-02 09:30:00,120.07,120.08,119.84,119.93,8570,2025-01-02 14:31:00+00:00,0,2025-01-02 09:30:00,2025-01-02 14:31:00+00:00,0.0,2025-01-02,09:30:00
1,RUN-2025A,ARVT,2025-01-02 09:31:00,119.93,119.98,119.35,119.39,45001,2025-01-02 14:32:01+00:00,1,2025-01-02 09:31:00,2025-01-02 14:32:00+00:00,1.0,2025-01-02,09:31:00
2,RUN-2025A,ARVT,2025-01-02 09:32:00,119.39,119.66,119.30,119.53,5825,2025-01-02 14:33:00+00:00,2,2025-01-02 09:32:00,2025-01-02 14:33:00+00:00,0.0,2025-01-02,09:32:00
3,RUN-2025A,ARVT,2025-01-02 09:33:00,119.53,119.57,119.23,119.35,8195,2025-01-02 14:34:00+00:00,3,2025-01-02 09:33:00,2025-01-02 14:34:00+00:00,0.0,2025-01-02,09:33:00
4,RUN-2025A,ARVT,2025-01-02 09:34:00,119.35,119.44,119.33,119.38,5631,2025-01-02 14:35:01+00:00,4,2025-01-02 09:34:00,2025-01-02 14:35:00+00:00,1.0,2025-01-02,09:34:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4271219,RUN-2025C,INDEX,2025-06-30 15:55:00,1234.18,1234.59,1234.17,1234.43,126336,2025-06-30 19:56:00+00:00,4271219,2025-06-30 15:55:00,2025-06-30 19:56:00+00:00,0.0,2025-06-30,15:55:00
4271220,RUN-2025C,INDEX,2025-06-30 15:56:00,1234.43,1235.04,1234.19,1235.00,135129,2025-06-30 19:57:02+00:00,4271220,2025-06-30 15:56:00,2025-06-30 19:57:00+00:00,2.0,2025-06-30,15:56:00
4271221,RUN-2025C,INDEX,2025-06-30 15:57:00,1235.00,1235.18,1234.12,1234.39,126898,2025-06-30 19:58:00+00:00,4271221,2025-06-30 15:57:00,2025-06-30 19:58:00+00:00,0.0,2025-06-30,15:57:00
4271222,RUN-2025C,INDEX,2025-06-30 15:58:00,1234.39,1234.42,1234.13,1234.37,115500,2025-06-30 19:59:02+00:00,4271222,2025-06-30 15:58:00,2025-06-30 19:59:00+00:00,2.0,2025-06-30,15:58:00


#### Cleansing Results

In [36]:
# Results
rejects = pd.concat(all_rejects, ignore_index=True)
clean = df

print(rejects["reason"].value_counts())

reason
DUPLICATE_BAR             15120
INVALID_OHLC               1282
MISSING_REQUIRED_FIELD      854
INVALID_NUMERIC_RANGE       853
MALFORMED_TIMESTAMP         342
UNKNOWN_RUN                 260
Name: count, dtype: int64


RUN-2025B appears twice in sim_runs; same config_version, but created_at differs. 
For the bar join, should retain one run record and exclude conflicting "created_at" as the relation is many_to_one

In [37]:
sim_runs_clean = (
    sim_runs_flat[["run_id", "config_version"]]
    .drop_duplicates(subset=["run_id", "config_version"])
)

In [38]:
bars_with_runs= bars_cleaned.merge(
    sim_runs_clean,
    on="run_id",
    how="left",
    validate="many_to_one"
)

In [39]:
print("bars before clean :", len(bars))
print("bars after clean:", len(bars_cleaned))
print("bars after merge: ", len(bars_with_runs))
print("unmatched bars:", bars_with_runs["config_version"].isna().sum())

bars before clean : 4276175
bars after clean: 4257598
bars after merge:  4257598
unmatched bars: 0


In [40]:
bars_with_runs.loc[
    bars_with_runs["config_version"].isna(),
    "run_id"
].drop_duplicates()

Series([], Name: run_id, dtype: str)

### Merge `bars_with_runs` with `exchange_calendar.csv` 

with `is_trading_day`= True and `bars_with_runs`.`trading_date`= `exchange_calendar`.`date` and `bars_with_runs`.`bar_time` is between `exchange_calendar`.`session_open` and `session_close` all as IN_SESSION ELSE OUTSIDE_SESSION

In [72]:
exchange_calendar = pd.read_csv(DATA_DIR / "exchange_calendar.csv")

In [89]:
exchange_calendar.head()

,date,is_trading_day,session_open,session_close,timezone,note
0,2025-01-02,True,09:30:00,16:00:00,America/New_York,NaN
1,2025-01-03,True,09:30:00,16:00:00,America/New_York,NaN
2,2025-01-06,True,09:30:00,16:00:00,America/New_York,NaN
3,2025-01-07,True,09:30:00,16:00:00,America/New_York,NaN
4,2025-01-08,True,09:30:00,16:00:00,America/New_York,NaN


In [88]:
print(exchange_calendar.dtypes)

date              object
is_trading_day      bool
session_open      object
session_close     object
timezone             str
note                 str
dtype: object


In [97]:
print(bars_with_runs.dtypes)

run_id                            str
ticker                            str
ts                                str
open                          float64
high                          float64
low                           float64
close                         float64
volume                          int64
received_at       datetime64[ms, UTC]
_source_row                     int64
ts_local               datetime64[us]
end_utc           datetime64[us, UTC]
lateness_s                    float64
trading_date                   object
bar_time                       object
config_version                    str
dtype: object


We should convert the `exchange_calendar` date column to datetime.date and `session_open` and `session_close` to datetime.time

In [75]:
exchange_calendar["date"] = pd.to_datetime(
    exchange_calendar["date"]
).dt.date

In [84]:
exchange_calendar["session_open"] = pd.to_datetime(
    exchange_calendar["session_open"], format="%H:%M"
).dt.time

In [87]:
exchange_calendar["session_close"] = pd.to_datetime(
    exchange_calendar["session_close"], format="%H:%M"
).dt.time

In [98]:
print(type(bars_with_runs["trading_date"].iloc[0]))
print(type(exchange_calendar["date"].iloc[0]))

print(
    repr(bars_with_runs["trading_date"].iloc[0]),
    repr(exchange_calendar["date"].iloc[0])
)

<class 'datetime.date'>
<class 'datetime.date'>
datetime.date(2025, 1, 2) datetime.date(2025, 1, 2)


In [90]:
print(exchange_calendar.dtypes)

date              object
is_trading_day      bool
session_open      object
session_close     object
timezone             str
note                 str
dtype: object


In [82]:
print(bars_with_runs.dtypes)

run_id                            str
ticker                            str
ts                                str
open                          float64
high                          float64
low                           float64
close                         float64
volume                          int64
received_at       datetime64[ms, UTC]
_source_row                     int64
ts_local               datetime64[us]
end_utc           datetime64[us, UTC]
lateness_s                    float64
trading_date                   object
bar_time                       object
config_version                    str
dtype: object


Check Nulls in `Exchange_calendar`

In [47]:
print("Distinct session_open values:")
display(exchange_calendar["session_open"].drop_duplicates().sort_values())

print("Distinct session_close values:")
display(exchange_calendar["session_close"].drop_duplicates().sort_values())

Distinct session_open values:


0     09:30:00
12         NaN
Name: session_open, dtype: str

Distinct session_close values:


75    13:00:00
0     16:00:00
12         NaN
Name: session_close, dtype: str

According to the calendar definition, `session_open` and `session_close` are nullable local session hours, empty when the market is closed.

So the NaN values are valid and expected, not necessarily data quality issues.

In [99]:
bars_with_runs.head()

The history saving thread hit an unexpected error (OperationalError('database or disk is full')).History will not be written to the database.


,run_id,ticker,ts,open,high,low,close,volume,received_at,_source_row,ts_local,end_utc,lateness_s,trading_date,bar_time,config_version
0,RUN-2025A,ARVT,2025-01-02 09:30:00,120.07,120.08,119.84,119.93,8570,2025-01-02 14:31:00+00:00,0,2025-01-02 09:30:00,2025-01-02 14:31:00+00:00,0.0,2025-01-02,09:30:00,1.4.0
1,RUN-2025A,ARVT,2025-01-02 09:31:00,119.93,119.98,119.35,119.39,45001,2025-01-02 14:32:01+00:00,1,2025-01-02 09:31:00,2025-01-02 14:32:00+00:00,1.0,2025-01-02,09:31:00,1.4.0
2,RUN-2025A,ARVT,2025-01-02 09:32:00,119.39,119.66,119.30,119.53,5825,2025-01-02 14:33:00+00:00,2,2025-01-02 09:32:00,2025-01-02 14:33:00+00:00,0.0,2025-01-02,09:32:00,1.4.0
3,RUN-2025A,ARVT,2025-01-02 09:33:00,119.53,119.57,119.23,119.35,8195,2025-01-02 14:34:00+00:00,3,2025-01-02 09:33:00,2025-01-02 14:34:00+00:00,0.0,2025-01-02,09:33:00,1.4.0
4,RUN-2025A,ARVT,2025-01-02 09:34:00,119.35,119.44,119.33,119.38,5631,2025-01-02 14:35:01+00:00,4,2025-01-02 09:34:00,2025-01-02 14:35:00+00:00,1.0,2025-01-02,09:34:00,1.4.0


In [49]:
# Check whether date is unique in the exchange calendar
duplicate_dates = exchange_calendar[
    exchange_calendar["date"].duplicated(keep=False)
]

display(duplicate_dates)

,date,is_trading_day,session_open,session_close,timezone,note


In [91]:
bars_with_calendar = bars_with_runs.merge(
    exchange_calendar,
    left_on="trading_date",
    right_on="date",
    how="left",
    validate="many_to_one"
)

In [92]:
print("Rows before merge:", len(bars_with_runs))
print("Rows after merge:", len(bars_with_calendar))

assert len(bars_with_calendar) == len(bars_with_runs), \
    "Unexpected row duplication after calendar merge!"

Rows before merge: 4257598
Rows after merge: 4257598


In [52]:
assert len(bars_with_calendar) == len(bars_with_runs), \
    "Unexpected row duplication after calendar merge!"

In [93]:
bars_with_calendar

,run_id,ticker,ts,open,high,low,close,volume,received_at,_source_row,...,lateness_s,trading_date,bar_time,config_version,date,is_trading_day,session_open,session_close,timezone,note
0,RUN-2025A,ARVT,2025-01-02 09:30:00,120.07,120.08,119.84,119.93,8570,2025-01-02 14:31:00+00:00,0,...,0.0,2025-01-02,09:30:00,1.4.0,2025-01-02,True,09:30:00,16:00:00,America/New_York,NaN
1,RUN-2025A,ARVT,2025-01-02 09:31:00,119.93,119.98,119.35,119.39,45001,2025-01-02 14:32:01+00:00,1,...,1.0,2025-01-02,09:31:00,1.4.0,2025-01-02,True,09:30:00,16:00:00,America/New_York,NaN
2,RUN-2025A,ARVT,2025-01-02 09:32:00,119.39,119.66,119.30,119.53,5825,2025-01-02 14:33:00+00:00,2,...,0.0,2025-01-02,09:32:00,1.4.0,2025-01-02,True,09:30:00,16:00:00,America/New_York,NaN
3,RUN-2025A,ARVT,2025-01-02 09:33:00,119.53,119.57,119.23,119.35,8195,2025-01-02 14:34:00+00:00,3,...,0.0,2025-01-02,09:33:00,1.4.0,2025-01-02,True,09:30:00,16:00:00,America/New_York,NaN
4,RUN-2025A,ARVT,2025-01-02 09:34:00,119.35,119.44,119.33,119.38,5631,2025-01-02 14:35:01+00:00,4,...,1.0,2025-01-02,09:34:00,1.4.0,2025-01-02,True,09:30:00,16:00:00,America/New_York,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4257593,RUN-2025C,INDEX,2025-06-30 15:55:00,1234.18,1234.59,1234.17,1234.43,126336,2025-06-30 19:56:00+00:00,4271219,...,0.0,2025-06-30,15:55:00,1.4.0,2025-06-30,True,09:30:00,16:00:00,America/New_York,NaN
4257594,RUN-2025C,INDEX,2025-06-30 15:56:00,1234.43,1235.04,1234.19,1235.00,135129,2025-06-30 19:57:02+00:00,4271220,...,2.0,2025-06-30,15:56:00,1.4.0,2025-06-30,True,09:30:00,16:00:00,America/New_York,NaN
4257595,RUN-2025C,INDEX,2025-06-30 15:57:00,1235.00,1235.18,1234.12,1234.39,126898,2025-06-30 19:58:00+00:00,4271221,...,0.0,2025-06-30,15:57:00,1.4.0,2025-06-30,True,09:30:00,16:00:00,America/New_York,NaN
4257596,RUN-2025C,INDEX,2025-06-30 15:58:00,1234.39,1234.42,1234.13,1234.37,115500,2025-06-30 19:59:02+00:00,4271222,...,2.0,2025-06-30,15:58:00,1.4.0,2025-06-30,True,09:30:00,16:00:00,America/New_York,NaN


In [94]:
bars_with_calendar["session_status"] = np.where(
    (
        bars_with_calendar["is_trading_day"].eq(True)
    ) &
    (
        bars_with_calendar["bar_time"] >= bars_with_calendar["session_open"]
    ) &
    (
        bars_with_calendar["bar_time"] <= bars_with_calendar["session_close"]
    ),
    "IN_SESSION",
    "OUTSIDE_SESSION"
)

In [ ]:
bars_with_calendar["session_status"].value_counts(dropna=False)

session_status
IN_SESSION         4240875
OUTSIDE_SESSION      16723
Name: count, dtype: int64

In [100]:
bars_with_calendar

,run_id,ticker,ts,open,high,low,close,volume,received_at,_source_row,...,trading_date,bar_time,config_version,date,is_trading_day,session_open,session_close,timezone,note,session_status
0,RUN-2025A,ARVT,2025-01-02 09:30:00,120.07,120.08,119.84,119.93,8570,2025-01-02 14:31:00+00:00,0,...,2025-01-02,09:30:00,1.4.0,2025-01-02,True,09:30:00,16:00:00,America/New_York,NaN,IN_SESSION
1,RUN-2025A,ARVT,2025-01-02 09:31:00,119.93,119.98,119.35,119.39,45001,2025-01-02 14:32:01+00:00,1,...,2025-01-02,09:31:00,1.4.0,2025-01-02,True,09:30:00,16:00:00,America/New_York,NaN,IN_SESSION
2,RUN-2025A,ARVT,2025-01-02 09:32:00,119.39,119.66,119.30,119.53,5825,2025-01-02 14:33:00+00:00,2,...,2025-01-02,09:32:00,1.4.0,2025-01-02,True,09:30:00,16:00:00,America/New_York,NaN,IN_SESSION
3,RUN-2025A,ARVT,2025-01-02 09:33:00,119.53,119.57,119.23,119.35,8195,2025-01-02 14:34:00+00:00,3,...,2025-01-02,09:33:00,1.4.0,2025-01-02,True,09:30:00,16:00:00,America/New_York,NaN,IN_SESSION
4,RUN-2025A,ARVT,2025-01-02 09:34:00,119.35,119.44,119.33,119.38,5631,2025-01-02 14:35:01+00:00,4,...,2025-01-02,09:34:00,1.4.0,2025-01-02,True,09:30:00,16:00:00,America/New_York,NaN,IN_SESSION
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4257593,RUN-2025C,INDEX,2025-06-30 15:55:00,1234.18,1234.59,1234.17,1234.43,126336,2025-06-30 19:56:00+00:00,4271219,...,2025-06-30,15:55:00,1.4.0,2025-06-30,True,09:30:00,16:00:00,America/New_York,NaN,IN_SESSION
4257594,RUN-2025C,INDEX,2025-06-30 15:56:00,1234.43,1235.04,1234.19,1235.00,135129,2025-06-30 19:57:02+00:00,4271220,...,2025-06-30,15:56:00,1.4.0,2025-06-30,True,09:30:00,16:00:00,America/New_York,NaN,IN_SESSION
4257595,RUN-2025C,INDEX,2025-06-30 15:57:00,1235.00,1235.18,1234.12,1234.39,126898,2025-06-30 19:58:00+00:00,4271221,...,2025-06-30,15:57:00,1.4.0,2025-06-30,True,09:30:00,16:00:00,America/New_York,NaN,IN_SESSION
4257596,RUN-2025C,INDEX,2025-06-30 15:58:00,1234.39,1234.42,1234.13,1234.37,115500,2025-06-30 19:59:02+00:00,4271222,...,2025-06-30,15:58:00,1.4.0,2025-06-30,True,09:30:00,16:00:00,America/New_York,NaN,IN_SESSION
